
# Getting Started: Lagrangian Stochastic (LS) Footprint Model

This notebook shows how to use the **`ls_footprint_model.py`** module to compute a simple
2‑D flux footprint from eddy‑covariance tower data. It uses the attached example
configuration (`US-UTE.ini`) and data file referenced therein.



## 1) Setup

This section adjusts `sys.path` so you can import from your local source tree, then imports
the LS model. If you've installed the package, the first `import` will also work.


In [ ]:

# --- Standard library
import os, sys, math, pathlib, warnings

# --- Add your local 'src' to the path (works for repo layouts like tests/notebooks)
# NOTE: In Jupyter, __file__ is usually undefined; we guard for that.
try:
    sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "../src")))
except NameError:
    # Fallback for notebooks: treat the current working directory as the repo root
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../src")))

# If you specifically need *exactly* this line somewhere (e.g., in tests), here it is:
# sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "../src")))

# --- Third‑party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Try imports from your package name (if installed) or the module directly
try:
    from fluxfootprints.ls_footprint_model import LSFootprintConfig, BackwardLSModel
except Exception:
    try:
        from ls_footprint_model import LSFootprintConfig, BackwardLSModel  # same directory/module name
    except Exception:
        # As a last resort, import straight from a file path next to this notebook
        import importlib.util, pathlib
        mod_path = pathlib.Path.cwd() / "ls_footprint_model.py"
        if not mod_path.exists():
            # Try the /mnt/data path used in this shared example
            mod_path = pathlib.Path("/mnt/data/ls_footprint_model.py")
        spec = importlib.util.spec_from_file_location("ls_footprint_model", str(mod_path))
        ls_mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(ls_mod)  # type: ignore
        LSFootprintConfig, BackwardLSModel = ls_mod.LSFootprintConfig, ls_mod.BackwardLSModel

plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore")
print("Imports ready.")



## 2) Load site configuration and data

We read `US-UTE.ini` to discover the time column, wind variables, and file names. Then we load the
CSV it references and make a minimal set of variables needed by the LS model.


In [ ]:

import configparser
from pathlib import Path

# Point to the example INI and CSV (adjust paths as needed)
ini_path = Path("US-UTE.ini")
if not ini_path.exists():
    ini_path = Path("/mnt/data/US-UTE.ini")  # fallback for this shared environment

cfg = configparser.ConfigParser()
cfg.read(ini_path)

# Pull the essentials
date_col = cfg.get("DATA", "datestring_col")
date_fmt = cfg.get("METADATA", "date_parser")
skiprows = cfg.getint("METADATA", "skiprows", fallback=0)
missing_val = cfg.getfloat("METADATA", "missing_data_value", fallback=-9999.0)

# Columns for wind speed and direction
ws_col = cfg.get("DATA", "wind_spd_col", fallback="WS")
wd_col = cfg.get("DATA", "wind_dir_col", fallback="WD")

# Resolve the climate/data CSV next to the INI, or in the same folder
csv_rel = cfg.get("METADATA", "climate_file_path")
csv_path = ini_path.parent / csv_rel
if not csv_path.exists():
    csv_path = Path("/mnt/data") / csv_rel  # fallback

# Load data
parse = lambda s: pd.to_datetime(s, format=date_fmt, errors="coerce")
df = pd.read_csv(csv_path, skiprows=skiprows)
df[date_col] = df[date_col].apply(parse)
df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()

# Basic cleaning for common missing value flags
df = df.replace({missing_val: np.nan, -9999: np.nan, -9999.0: np.nan})

print(f"Loaded {len(df):,} rows from:", csv_path)
df.head()



## 3) Choose a time slice and prepare LS model inputs

The LS model needs:
- `zm` (receptor height, m)
- `ustar` (friction velocity, m s⁻¹)
- `L` (Monin–Obukhov length, m) — **not used** in this minimal implementation but kept for extension
- `h` (boundary‑layer height, m)
- `wind_dir_deg` (meteorological *from* direction, degrees)
- surface roughness `z0` and a numerical grid/domain

If your dataset has a `USTAR` column, we use that. Otherwise we estimate \(u_*\) from wind speed
by the neutral log‑law (requires assumptions for `zm` and `z0`).


In [ ]:

KAPPA = 0.4

# Assumptions you can tune for your site
zm_default = 4.0   # receptor height [m]
z0_default = 0.1   # roughness length [m]
h_default  = 1000.0
L_default  = -50.0  # kept for API compatibility; not used in the core model

# Pick a representative timestamp with non‑missing wind + speed
needed = [wd_col]
if "USTAR" in df.columns:
    needed.append("USTAR")
else:
    needed.append(ws_col)

sub = df.dropna(subset=needed).copy()
if sub.empty:
    raise RuntimeError("No rows with required variables present. Check column names and missing values.")

row = sub.iloc[0]  # take the first valid row; feel free to select by date/time

# Derive ustar if needed
if "USTAR" in df.columns and pd.notna(row.get("USTAR")):
    ustar = float(row["USTAR"])
else:
    # Estimate u* with neutral log profile: u* = κ U / ln(zm / z0)
    U = float(row[ws_col])
    ustar = KAPPA * U / math.log(zm_default / z0_default)

wind_dir_deg = float(row[wd_col])
zm, z0, h, L = zm_default, z0_default, h_default, L_default

print(f"Selected time: {row.name}")
print(f"u* = {ustar:.3f} m s⁻1 | wind_dir = {wind_dir_deg:.1f}° | zm={zm} m, z0={z0} m, h={h} m")



## 4) Run the model


In [ ]:

cfg = LSFootprintConfig(
    zm=zm,
    ustar=ustar,
    L=L,
    h=h,
    wind_dir_deg=wind_dir_deg,
    z0=z0,
    n_particles=20_000,   # increase for smoother fields
    dt=0.25,
    t_max=600.0,
    domain=(2000.0, 2000.0),
    dx=20.0,
    dy=20.0,
    seed=42,              # reproducible runs
)
model = BackwardLSModel(cfg)
model.run()

x, y, F = model.footprint()
xx, fx = model.crosswind_integrated()

F.min(), F.max(), F.sum()



## 5) Visualize the footprint


In [ ]:

plt.figure(figsize=(6, 5))
plt.pcolormesh(y, x, F, shading="auto")
plt.colorbar(label="Footprint weight (m$^{-2}$)")
plt.xlabel("Cross‑wind distance y (m)")
plt.ylabel("Upwind distance −x (m)")
plt.title("2‑D Lagrangian flux footprint")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(-xx, fx)  # positive axis for downwind distance
plt.xlabel("Upwind distance (m)")
plt.ylabel("f(x) (m$^{-1}$)")
plt.title("Cross‑wind integrated footprint")
plt.grid(True)
plt.tight_layout()
plt.show()



## 6) Optional: 80% source area contour

Below we compute a density threshold such that the **integral** of the footprint inside the contour
is ~80% of the total. We then plot that as a single contour line.


In [ ]:

# Convert density (m^-2) to probability mass per cell
cell_mass = F * cfg.dx * cfg.dy
flat = cell_mass.ravel()
order = np.argsort(flat)[::-1]
csum = np.cumsum(flat[order])

# Find threshold where cumulative mass first exceeds 0.8
idx_thr = int(np.searchsorted(csum, 0.8))
thr_mass = flat[order][idx_thr]
thr_density = thr_mass / (cfg.dx * cfg.dy)

plt.figure(figsize=(6, 5))
plt.pcolormesh(y, x, F, shading="auto")
plt.colorbar(label="Footprint weight (m$^{-2}$)")
cc = plt.contour(y, x, F, levels=[thr_density])
plt.clabel(cc, fmt="80% area", inline=True, fontsize=8)
plt.xlabel("Cross‑wind distance y (m)")
plt.ylabel("Upwind distance −x (m)")
plt.title("Footprint with ~80% source area contour")
plt.tight_layout()
plt.show()



## 7) Save outputs (optional)


In [ ]:

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

# Save grids
np.save(out_dir / "footprint_density.npy", F)
np.savetxt(out_dir / "footprint_density.csv", F, delimiter=",")
np.savetxt(out_dir / "x_centers_m.csv", x, delimiter=",")
np.savetxt(out_dir / "y_centers_m.csv", y, delimiter=",")

# Save figures
plt.figure(figsize=(6, 5))
plt.pcolormesh(y, x, F, shading="auto")
plt.colorbar(label="Footprint weight (m$^{-2}$)")
plt.xlabel("Cross‑wind distance y (m)")
plt.ylabel("Upwind distance −x (m)")
plt.title("2‑D LS Footprint")
plt.tight_layout()
plt.savefig(out_dir / "footprint_2d.png", dpi=200)
plt.close()

plt.figure(figsize=(6, 4))
plt.plot(-xx, fx)
plt.xlabel("Upwind distance (m)")
plt.ylabel("f(x) (m$^{-1}$)")
plt.title("Cross‑wind integrated footprint")
plt.grid(True)
plt.tight_layout()
plt.savefig(out_dir / "footprint_fx.png", dpi=200)
plt.close()

print("Saved outputs to:", out_dir.resolve())



## 8) Tips & next steps

- Increase `n_particles` for smoother fields (e.g., 50,000–200,000).
- Adjust `domain`, `dx` and `dy` to control the spatial extent and resolution.
- If you have measured `USTAR`, use it directly; otherwise, provide a better local estimate for `z0` and set your actual `zm`.
- The current `L` parameter is kept for future extensions (e.g., stability‑dependent turbulence); it is not used in the core update step here.
- For climatological footprints, repeat runs across many time steps and average the resulting `F` fields (watch memory and runtime).
